In [1]:
%pip install -U osmnx geopandas networkx pyogrio

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys
import time
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import box

# ============================================================
# ROADFLOOD-VLM
# Notebook 04: Transportation Knowledge Graph Generation
# ============================================================

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MAP_DIR = OUTPUT_DIR / "maps"

TKG_DIR = PROCESSED_DIR / "transportation_knowledge_graphs"
GRAPHML_DIR = TKG_DIR / "graphml"
GPKG_DIR = TKG_DIR / "geopackages"
FAILED_DIR = TKG_DIR / "failed"

for directory in [
    INTERIM_DIR,
    PROCESSED_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    MAP_DIR,
    TKG_DIR,
    GRAPHML_DIR,
    GPKG_DIR,
    FAILED_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

ranking_path = (
    TABLE_DIR
    / "roadflood_vlm_preliminary_scene_ranking.csv"
)

if not ranking_path.exists():
    raise FileNotFoundError(
        "Scene ranking table was not found. "
        "Run Notebook 03 first."
    )

# OSMnx configuration
ox.settings.use_cache = True
ox.settings.cache_folder = str(
    INTERIM_DIR / "osmnx_cache"
)
ox.settings.log_console = False
ox.settings.requests_timeout = 180

print("=" * 72)
print("ROADFLOOD-VLM TRANSPORTATION KNOWLEDGE GRAPH")
print("=" * 72)

print(f"\nProject root     : {PROJECT_ROOT}")
print(f"Ranking table    : {ranking_path}")
print(f"OSMnx version    : {ox.__version__}")
print(f"GeoPandas version: {gpd.__version__}")
print(f"NetworkX version : {nx.__version__}")
print(f"Python version   : {sys.version.split()[0]}")

print("\n" + "=" * 72)
print("NOTEBOOK 04 INITIALIZATION COMPLETE")
print("=" * 72)

ROADFLOOD-VLM TRANSPORTATION KNOWLEDGE GRAPH

Project root     : /home/adjeiowusu1/myproject/ResilientVLM
Ranking table    : /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_preliminary_scene_ranking.csv
OSMnx version    : 2.0.7
GeoPandas version: 1.1.4
NetworkX version : 3.4.2
Python version   : 3.10.12

NOTEBOOK 04 INITIALIZATION COMPLETE


In [3]:
# ============================================================
# CELL 3: LOAD AND SELECT TOP-RANKED SCENES
# ============================================================

print("=" * 72)
print("TOP-RANKED TRANSPORTATION SCENE SELECTION")
print("=" * 72)

ranking_df = pd.read_csv(ranking_path)

required_columns = [
    "scene_id",
    "country_prefix",
    "split",
    "eligible",
    "quality_rank",
    "left",
    "bottom",
    "right",
    "top",
    "valid_pct",
    "flood_pct_valid",
    "permanent_water_pct",
    "preliminary_suitability_score",
]

missing_columns = [
    column
    for column in required_columns
    if column not in ranking_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Convert eligible safely in case CSV stores it as text
if ranking_df["eligible"].dtype == object:
    ranking_df["eligible"] = (
        ranking_df["eligible"]
        .astype(str)
        .str.lower()
        .map({"true": True, "false": False})
    )

TOP_N = 30

top_scenes_df = (
    ranking_df.loc[ranking_df["eligible"] == True]
    .sort_values(
        "preliminary_suitability_score",
        ascending=False,
    )
    .head(TOP_N)
    .copy()
    .reset_index(drop=True)
)

top_scenes_df["prototype_order"] = (
    np.arange(1, len(top_scenes_df) + 1)
)

print(f"\nEligible scenes available: "
      f"{ranking_df['eligible'].sum():,}")

print(f"Prototype scenes selected: {len(top_scenes_df):,}")

print("\nScenes by split:")
print(
    top_scenes_df["split"]
    .value_counts()
    .to_string()
)

print("\nScenes by country or region:")
print(
    top_scenes_df["country_prefix"]
    .value_counts()
    .to_string()
)

display_columns = [
    "prototype_order",
    "quality_rank",
    "scene_id",
    "country_prefix",
    "split",
    "flood_pct_valid",
    "preliminary_suitability_score",
]

print("\nSELECTED SCENES")
print("-" * 72)

print(
    top_scenes_df[display_columns]
    .round(3)
    .to_string(index=False)
)

top_scene_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_scenes.csv"
)

top_scenes_df.to_csv(
    top_scene_path,
    index=False,
)

print(f"\nSaved to: {top_scene_path}")

print("\n" + "=" * 72)
print("TOP-30 SCENE SELECTION COMPLETE")
print("=" * 72)

TOP-RANKED TRANSPORTATION SCENE SELECTION

Eligible scenes available: 208
Prototype scenes selected: 30

Scenes by split:
split
train         22
validation     5
test           3

Scenes by country or region:
country_prefix
India        10
USA           5
Spain         3
Somalia       3
Paraguay      3
Nigeria       2
Sri-Lanka     2
Mekong        1
Ghana         1

SELECTED SCENES
------------------------------------------------------------------------
 prototype_order  quality_rank        scene_id country_prefix      split  flood_pct_valid  preliminary_suitability_score
               1             1   Spain_8565131          Spain validation           19.977                         93.567
               2             2    India_804466          India      train           19.161                         92.016
               3             3   India_1050276          India validation           19.204                         89.661
               4             4 Nigeria_1095404        Nige

In [4]:
# ============================================================
# CELL 4: BUILD AND BUFFER SCENE EXTENTS
# ============================================================

print("=" * 72)
print("SCENE BOUNDARY GENERATION")
print("=" * 72)

scene_geometry = [
    box(
        row["left"],
        row["bottom"],
        row["right"],
        row["top"],
    )
    for _, row in top_scenes_df.iterrows()
]

scene_bounds_gdf = gpd.GeoDataFrame(
    top_scenes_df.copy(),
    geometry=scene_geometry,
    crs="EPSG:4326",
)

if not scene_bounds_gdf.geometry.is_valid.all():
    raise ValueError(
        "One or more scene geometries are invalid."
    )

BUFFER_METERS = 250

buffered_records = []

for _, row in scene_bounds_gdf.iterrows():

    single_scene = gpd.GeoDataFrame(
        [row],
        geometry="geometry",
        crs="EPSG:4326",
    )

    local_crs = single_scene.estimate_utm_crs()

    if local_crs is None:
        raise ValueError(
            f"Could not determine projected CRS "
            f"for {row['scene_id']}."
        )

    projected = single_scene.to_crs(local_crs)

    buffered_projected = projected.copy()
    buffered_projected["geometry"] = (
        projected.geometry.buffer(BUFFER_METERS)
    )

    buffered_wgs84 = (
        buffered_projected
        .to_crs("EPSG:4326")
    )

    buffered_geometry = (
        buffered_wgs84.geometry.iloc[0]
    )

    minx, miny, maxx, maxy = (
        buffered_geometry.bounds
    )

    buffered_records.append(
        {
            **row.drop(labels="geometry").to_dict(),
            "local_projected_crs": local_crs.to_string(),
            "buffer_meters": BUFFER_METERS,
            "osm_left": minx,
            "osm_bottom": miny,
            "osm_right": maxx,
            "osm_top": maxy,
            "geometry": buffered_geometry,
        }
    )

buffered_bounds_gdf = gpd.GeoDataFrame(
    buffered_records,
    geometry="geometry",
    crs="EPSG:4326",
)

print(f"\nScenes processed : {len(buffered_bounds_gdf):,}")
print(f"Buffer distance  : {BUFFER_METERS:,} meters")

print("\nBOUND VALIDATION")
print("-" * 72)

print(
    buffered_bounds_gdf[
        [
            "scene_id",
            "osm_left",
            "osm_bottom",
            "osm_right",
            "osm_top",
            "local_projected_crs",
        ]
    ]
    .round(6)
    .head(10)
    .to_string(index=False)
)

bounds_path = (
    GPKG_DIR
    / "roadflood_vlm_top30_scene_bounds.gpkg"
)

buffered_bounds_gdf.to_file(
    bounds_path,
    layer="buffered_scene_bounds",
    driver="GPKG",
)

print(f"\nSaved to: {bounds_path}")

print("\n" + "=" * 72)
print("SCENE BOUNDARY GENERATION COMPLETE")
print("=" * 72)

SCENE BOUNDARY GENERATION



Scenes processed : 30
Buffer distance  : 250 meters

BOUND VALIDATION
------------------------------------------------------------------------
       scene_id   osm_left  osm_bottom  osm_right    osm_top local_projected_crs
  Spain_8565131  -0.830742   38.172554  -0.779039  38.223052          EPSG:32630
   India_804466  92.812872   26.076195  92.863867  26.126703          EPSG:32646
  India_1050276  92.490914   26.168182  92.541913  26.218691          EPSG:32646
Nigeria_1095404   5.654960    8.736552   5.705495   8.787064          EPSG:32631
     USA_217598 -95.025938   38.448517 -94.974212  38.499015          EPSG:32615
 Somalia_322855  45.577549    3.125313  45.628043   3.175830          EPSG:32638
   Mekong_16233 105.783310   12.140087 105.833900  12.190602          EPSG:32648
   India_943439  93.318792   26.582126  93.369809  26.632634          EPSG:32646
Paraguay_305760 -57.034713  -24.746891 -56.983775 -24.696382          EPSG:32721
     USA_170264 -95.301906   38.586498 -95.250


Saved to: /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge_graphs/geopackages/roadflood_vlm_top30_scene_bounds.gpkg

SCENE BOUNDARY GENERATION COMPLETE


In [5]:
# ============================================================
# CELL 5: TEST OSM ROAD EXTRACTION ON ONE SCENE
# ============================================================

print("=" * 72)
print("SINGLE-SCENE OSM EXTRACTION TEST")
print("=" * 72)

test_scene = buffered_bounds_gdf.iloc[0]

test_scene_id = test_scene["scene_id"]

bbox = (
    float(test_scene["osm_left"]),
    float(test_scene["osm_bottom"]),
    float(test_scene["osm_right"]),
    float(test_scene["osm_top"]),
)

print(f"\nTest scene : {test_scene_id}")
print(f"Bounding box: {bbox}")

start_time = time.time()

try:
    test_graph = ox.graph.graph_from_bbox(
        bbox=bbox,
        network_type="drive",
        simplify=True,
        retain_all=True,
        truncate_by_edge=True,
    )

    elapsed_seconds = time.time() - start_time

    test_nodes_gdf, test_edges_gdf = (
        ox.convert.graph_to_gdfs(
            test_graph,
            nodes=True,
            edges=True,
        )
    )

    print("\nEXTRACTION SUCCESSFUL")
    print("-" * 72)

    print(
        f"Nodes              : "
        f"{len(test_nodes_gdf):,}"
    )

    print(
        f"Directed edges     : "
        f"{len(test_edges_gdf):,}"
    )

    print(
        f"Connected components: "
        f"{nx.number_weakly_connected_components(test_graph):,}"
    )

    print(
        f"Processing time    : "
        f"{elapsed_seconds:,.2f} seconds"
    )

    print("\nEdge columns:")
    print(sorted(test_edges_gdf.columns.tolist()))

except Exception as exc:
    test_graph = None
    test_nodes_gdf = None
    test_edges_gdf = None

    print("\nEXTRACTION FAILED")
    print("-" * 72)
    print(type(exc).__name__)
    print(str(exc))

SINGLE-SCENE OSM EXTRACTION TEST

Test scene : Spain_8565131
Bounding box: (-0.8307421473940633, 38.172554160300336, -0.7790388995383078, 38.22305219291402)



EXTRACTION SUCCESSFUL
------------------------------------------------------------------------
Nodes              : 449
Directed edges     : 902
Connected components: 9
Processing time    : 0.64 seconds

Edge columns:
['access', 'bridge', 'geometry', 'highway', 'junction', 'lanes', 'length', 'maxspeed', 'name', 'oneway', 'osmid', 'ref', 'reversed', 'tunnel']


In [6]:
# ============================================================
# CELL 6: INSPECT TEST NETWORK ATTRIBUTES
# ============================================================

if test_graph is None:
    raise RuntimeError(
        "The single-scene OSM test failed. "
        "Do not continue until Cell 5 succeeds."
    )

inspection_columns = [
    column
    for column in [
        "osmid",
        "name",
        "highway",
        "length",
        "lanes",
        "maxspeed",
        "bridge",
        "tunnel",
        "oneway",
        "geometry",
    ]
    if column in test_edges_gdf.columns
]

print("=" * 72)
print("TEST NETWORK ATTRIBUTE INSPECTION")
print("=" * 72)

print(
    test_edges_gdf[
        inspection_columns
    ]
    .head(15)
    .to_string()
)

print("\nHIGHWAY CLASS COUNTS")
print("-" * 72)

if "highway" in test_edges_gdf.columns:

    highway_counts = (
        test_edges_gdf["highway"]
        .astype(str)
        .value_counts()
        .head(20)
    )

    print(highway_counts.to_string())

else:
    print("No highway field was returned.")

print("\n" + "=" * 72)
print("TEST NETWORK INSPECTION COMPLETE")
print("=" * 72)

TEST NETWORK ATTRIBUTE INSPECTION
                                                                                        osmid                                                       name       highway       length   lanes maxspeed bridge tunnel  oneway                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [7]:
# ============================================================
# CELL 7: HELPER FUNCTIONS FOR TRANSPORTATION KNOWLEDGE GRAPHS
# ============================================================

import ast
import math
import re
from collections import Counter


def normalize_osm_value(value):
    """
    Convert OSM scalar/list-like values to a clean string.
    """
    if value is None:
        return None

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, (list, tuple, set)):
        cleaned = [
            str(item).strip()
            for item in value
            if item is not None
        ]
        return "|".join(sorted(set(cleaned))) if cleaned else None

    return str(value).strip()


def first_numeric_value(value):
    """
    Extract the first numeric value from an OSM attribute such as:
    '50', '50 mph', ['2', '3'], or '2;3'.
    """
    if value is None:
        return np.nan

    if isinstance(value, float) and np.isnan(value):
        return np.nan

    if isinstance(value, (list, tuple, set)):
        for item in value:
            result = first_numeric_value(item)
            if not np.isnan(result):
                return result
        return np.nan

    match = re.search(r"[-+]?\d*\.?\d+", str(value))

    return float(match.group()) if match else np.nan


def normalize_boolean_tag(value):
    """
    Normalize bridge/tunnel-type OSM values to Boolean form.
    """
    if value is None:
        return False

    if isinstance(value, float) and np.isnan(value):
        return False

    if isinstance(value, (list, tuple, set)):
        return any(normalize_boolean_tag(item) for item in value)

    text = str(value).strip().lower()

    false_values = {
        "",
        "no",
        "false",
        "0",
        "none",
        "nan",
    }

    return text not in false_values


def primary_highway_class(value):
    """
    Return one primary roadway class from scalar or list-like values.
    """
    class_priority = [
        "motorway",
        "trunk",
        "primary",
        "secondary",
        "tertiary",
        "unclassified",
        "residential",
        "service",
        "living_street",
        "road",
    ]

    if value is None:
        return "unknown"

    if isinstance(value, float) and np.isnan(value):
        return "unknown"

    values = (
        list(value)
        if isinstance(value, (list, tuple, set))
        else [value]
    )

    cleaned = [
        str(item).strip().lower()
        for item in values
    ]

    for road_class in class_priority:
        if road_class in cleaned:
            return road_class

    return cleaned[0] if cleaned else "unknown"


def classify_road_importance(highway_class):
    """
    Convert OSM roadway class to an ordinal importance level.
    """
    importance_map = {
        "motorway": 5,
        "trunk": 5,
        "primary": 4,
        "secondary": 3,
        "tertiary": 2,
        "unclassified": 1,
        "residential": 1,
        "service": 0,
        "living_street": 0,
        "road": 0,
        "unknown": 0,
    }

    return importance_map.get(highway_class, 0)


def safe_graph_density(graph):
    """
    Compute graph density safely.
    """
    if graph.number_of_nodes() < 2:
        return 0.0

    return nx.density(graph)


def build_scene_network_metrics(
    graph,
    nodes_gdf,
    edges_gdf,
    scene_id,
    scene_area_km2,
):
    """
    Build scene-level transportation-network statistics.
    """

    undirected_graph = ox.convert.to_undirected(graph)

    node_count = graph.number_of_nodes()
    directed_edge_count = graph.number_of_edges()
    undirected_edge_count = undirected_graph.number_of_edges()

    weak_components = nx.number_weakly_connected_components(graph)
    strong_components = nx.number_strongly_connected_components(graph)

    if node_count > 0:
        largest_component_nodes = max(
            len(component)
            for component in nx.weakly_connected_components(graph)
        )
    else:
        largest_component_nodes = 0

    total_length_m = float(
        pd.to_numeric(
            edges_gdf["length"],
            errors="coerce",
        )
        .fillna(0)
        .sum()
    )

    road_density_km_per_km2 = (
        (total_length_m / 1000) / scene_area_km2
        if scene_area_km2 > 0
        else 0.0
    )

    degrees = dict(undirected_graph.degree())

    degree_values = list(degrees.values())

    mean_degree = (
        float(np.mean(degree_values))
        if degree_values
        else 0.0
    )

    intersection_count = int(
        sum(degree >= 3 for degree in degree_values)
    )

    dead_end_count = int(
        sum(degree == 1 for degree in degree_values)
    )

    highway_counts = (
        edges_gdf["highway_class"]
        .value_counts()
        .to_dict()
    )

    bridge_edge_count = int(
        edges_gdf["is_bridge"].sum()
    )

    tunnel_edge_count = int(
        edges_gdf["is_tunnel"].sum()
    )

    named_edge_count = int(
        edges_gdf["name_clean"].notna().sum()
    )

    lane_coverage_pct = (
        100
        * edges_gdf["lanes_numeric"].notna().mean()
        if len(edges_gdf) > 0
        else 0.0
    )

    speed_coverage_pct = (
        100
        * edges_gdf["maxspeed_numeric"].notna().mean()
        if len(edges_gdf) > 0
        else 0.0
    )

    return {
        "scene_id": scene_id,
        "node_count": node_count,
        "directed_edge_count": directed_edge_count,
        "undirected_edge_count": undirected_edge_count,
        "weak_component_count": weak_components,
        "strong_component_count": strong_components,
        "largest_component_nodes": largest_component_nodes,
        "largest_component_share": (
            largest_component_nodes / node_count
            if node_count > 0
            else 0.0
        ),
        "total_road_length_km": total_length_m / 1000,
        "scene_area_km2": scene_area_km2,
        "road_density_km_per_km2": road_density_km_per_km2,
        "mean_node_degree": mean_degree,
        "intersection_count": intersection_count,
        "intersection_density_per_km2": (
            intersection_count / scene_area_km2
            if scene_area_km2 > 0
            else 0.0
        ),
        "dead_end_count": dead_end_count,
        "graph_density": safe_graph_density(undirected_graph),
        "road_class_count": edges_gdf["highway_class"].nunique(),
        "bridge_edge_count": bridge_edge_count,
        "tunnel_edge_count": tunnel_edge_count,
        "named_edge_count": named_edge_count,
        "lane_coverage_pct": lane_coverage_pct,
        "speed_coverage_pct": speed_coverage_pct,
        "motorway_edge_count": highway_counts.get("motorway", 0),
        "trunk_edge_count": highway_counts.get("trunk", 0),
        "primary_edge_count": highway_counts.get("primary", 0),
        "secondary_edge_count": highway_counts.get("secondary", 0),
        "tertiary_edge_count": highway_counts.get("tertiary", 0),
        "residential_edge_count": highway_counts.get("residential", 0),
        "unclassified_edge_count": highway_counts.get("unclassified", 0),
    }


print("=" * 72)
print("TRANSPORTATION HELPER FUNCTIONS READY")
print("=" * 72)

TRANSPORTATION HELPER FUNCTIONS READY


In [8]:
# ============================================================
# CELL 8: BATCH TRANSPORTATION KNOWLEDGE GRAPH GENERATION
# ============================================================

from tqdm.auto import tqdm
import traceback

print("=" * 72)
print("TOP-30 TRANSPORTATION KNOWLEDGE GRAPH GENERATION")
print("=" * 72)

MAX_RETRIES = 3
RETRY_WAIT_SECONDS = 15
REQUEST_PAUSE_SECONDS = 3

scene_metric_records = []
processing_records = []

for _, scene in tqdm(
    buffered_bounds_gdf.iterrows(),
    total=len(buffered_bounds_gdf),
    desc="Processing transportation scenes",
):

    scene_id = scene["scene_id"]

    graphml_path = (
        GRAPHML_DIR
        / f"{scene_id}_transportation_graph.graphml"
    )

    gpkg_path = (
        GPKG_DIR
        / f"{scene_id}_transportation_knowledge.gpkg"
    )

    start_time = time.time()

    print("\n" + "-" * 72)
    print(f"Processing: {scene_id}")

    # --------------------------------------------------------
    # Skip completed scenes
    # --------------------------------------------------------

    if graphml_path.exists() and gpkg_path.exists():

        print("Existing outputs found. Loading saved graph.")

        try:
            graph = ox.io.load_graphml(graphml_path)

            nodes_gdf, edges_gdf = (
                ox.convert.graph_to_gdfs(graph)
            )

            status = "LOADED_EXISTING"

        except Exception as exc:
            print(
                "Existing output could not be loaded. "
                "The scene will be regenerated."
            )
            graph = None
            status = "REGENERATE"

    else:
        graph = None
        status = "NEW"

    # --------------------------------------------------------
    # Download graph when necessary
    # --------------------------------------------------------

    if graph is None:

        bbox = (
            float(scene["osm_left"]),
            float(scene["osm_bottom"]),
            float(scene["osm_right"]),
            float(scene["osm_top"]),
        )

        last_error = None

        for attempt in range(1, MAX_RETRIES + 1):

            try:

                print(
                    f"OSM request attempt "
                    f"{attempt}/{MAX_RETRIES}"
                )

                graph = ox.graph.graph_from_bbox(
                    bbox=bbox,
                    network_type="drive",
                    simplify=True,
                    retain_all=True,
                    truncate_by_edge=True,
                )

                nodes_gdf, edges_gdf = (
                    ox.convert.graph_to_gdfs(
                        graph,
                        nodes=True,
                        edges=True,
                    )
                )

                status = "DOWNLOADED"
                last_error = None
                break

            except Exception as exc:

                last_error = exc

                print(
                    f"Attempt {attempt} failed: "
                    f"{type(exc).__name__}: {exc}"
                )

                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_WAIT_SECONDS)

        if graph is None:

            elapsed_seconds = time.time() - start_time

            error_text = (
                f"{type(last_error).__name__}: "
                f"{last_error}"
            )

            processing_records.append(
                {
                    "scene_id": scene_id,
                    "status": "FAILED",
                    "processing_seconds": elapsed_seconds,
                    "error": error_text,
                }
            )

            failure_path = (
                FAILED_DIR
                / f"{scene_id}_failure.txt"
            )

            failure_path.write_text(
                error_text,
                encoding="utf-8",
            )

            print(f"Scene failed: {error_text}")
            continue

    # --------------------------------------------------------
    # Clean edge attributes
    # --------------------------------------------------------

    edges_gdf = edges_gdf.copy()
    nodes_gdf = nodes_gdf.copy()

    edges_gdf["scene_id"] = scene_id

    edges_gdf["highway_clean"] = (
        edges_gdf["highway"]
        .apply(normalize_osm_value)
        if "highway" in edges_gdf.columns
        else None
    )

    edges_gdf["highway_class"] = (
        edges_gdf["highway"]
        .apply(primary_highway_class)
        if "highway" in edges_gdf.columns
        else "unknown"
    )

    edges_gdf["road_importance"] = (
        edges_gdf["highway_class"]
        .apply(classify_road_importance)
    )

    edges_gdf["name_clean"] = (
        edges_gdf["name"]
        .apply(normalize_osm_value)
        if "name" in edges_gdf.columns
        else None
    )

    edges_gdf["lanes_numeric"] = (
        edges_gdf["lanes"]
        .apply(first_numeric_value)
        if "lanes" in edges_gdf.columns
        else np.nan
    )

    edges_gdf["maxspeed_numeric"] = (
        edges_gdf["maxspeed"]
        .apply(first_numeric_value)
        if "maxspeed" in edges_gdf.columns
        else np.nan
    )

    edges_gdf["is_bridge"] = (
        edges_gdf["bridge"]
        .apply(normalize_boolean_tag)
        if "bridge" in edges_gdf.columns
        else False
    )

    edges_gdf["is_tunnel"] = (
        edges_gdf["tunnel"]
        .apply(normalize_boolean_tag)
        if "tunnel" in edges_gdf.columns
        else False
    )

    edges_gdf["is_oneway"] = (
        edges_gdf["oneway"].astype(bool)
        if "oneway" in edges_gdf.columns
        else False
    )

    edges_gdf["length_m"] = pd.to_numeric(
        edges_gdf["length"],
        errors="coerce",
    )

    # --------------------------------------------------------
    # Compute node topology
    # --------------------------------------------------------

    undirected_graph = ox.convert.to_undirected(graph)

    degree_dict = dict(undirected_graph.degree())

    nodes_gdf["scene_id"] = scene_id

    nodes_gdf["node_degree"] = (
        nodes_gdf.index.map(degree_dict)
        .fillna(0)
        .astype(int)
    )

    nodes_gdf["is_intersection"] = (
        nodes_gdf["node_degree"] >= 3
    )

    nodes_gdf["is_dead_end"] = (
        nodes_gdf["node_degree"] == 1
    )

    # --------------------------------------------------------
    # Estimate scene area in projected coordinates
    # --------------------------------------------------------

    scene_polygon = gpd.GeoDataFrame(
        [{"geometry": scene.geometry}],
        crs="EPSG:4326",
    )

    local_crs = scene["local_projected_crs"]

    scene_area_km2 = (
        scene_polygon
        .to_crs(local_crs)
        .geometry.area.iloc[0]
        / 1_000_000
    )

    # --------------------------------------------------------
    # Create scene-level metadata
    # --------------------------------------------------------

    scene_metrics = build_scene_network_metrics(
        graph=graph,
        nodes_gdf=nodes_gdf,
        edges_gdf=edges_gdf,
        scene_id=scene_id,
        scene_area_km2=scene_area_km2,
    )

    scene_metrics.update(
        {
            "country_prefix": scene["country_prefix"],
            "split": scene["split"],
            "quality_rank": scene["quality_rank"],
            "valid_pct": scene["valid_pct"],
            "flood_pct_valid": scene["flood_pct_valid"],
            "permanent_water_pct": (
                scene["permanent_water_pct"]
            ),
            "preliminary_suitability_score": (
                scene[
                    "preliminary_suitability_score"
                ]
            ),
        }
    )

    scene_metric_records.append(scene_metrics)

    metadata_gdf = gpd.GeoDataFrame(
        [scene_metrics],
        geometry=[scene.geometry],
        crs="EPSG:4326",
    )

    # --------------------------------------------------------
    # Save outputs
    # --------------------------------------------------------

    ox.io.save_graphml(
        graph,
        filepath=graphml_path,
    )

    nodes_gdf.to_file(
        gpkg_path,
        layer="nodes",
        driver="GPKG",
    )

    edges_gdf.to_file(
        gpkg_path,
        layer="roads",
        driver="GPKG",
    )

    metadata_gdf.to_file(
        gpkg_path,
        layer="metadata",
        driver="GPKG",
    )

    elapsed_seconds = time.time() - start_time

    processing_records.append(
        {
            "scene_id": scene_id,
            "status": status,
            "processing_seconds": elapsed_seconds,
            "error": None,
        }
    )

    print(
        f"Completed: {len(nodes_gdf):,} nodes, "
        f"{len(edges_gdf):,} directed edges, "
        f"{scene_metrics['total_road_length_km']:.2f} km"
    )

    time.sleep(REQUEST_PAUSE_SECONDS)

print("\n" + "=" * 72)
print("TOP-30 TRANSPORTATION GRAPH PROCESSING COMPLETE")
print("=" * 72)

/home/adjeiowusu1/myproject/ResilientVLM/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TOP-30 TRANSPORTATION KNOWLEDGE GRAPH GENERATION


Processing transportation scenes:   0%|          | 0/30 [00:00<?, ?it/s]


------------------------------------------------------------------------
Processing: Spain_8565131
Existing outputs found. Loading saved graph.


Completed: 449 nodes, 902 directed edges, 224.28 km


...


------------------------------------------------------------------------
Processing: India_773682
Existing outputs found. Loading saved graph.


Completed: 83 nodes, 212 directed edges, 106.80 km


Processing transportation scenes: 100%|██████████| 30/30 [04:39<00:00, 12.09s/it]

Processing transportation scenes: 100%|██████████| 30/30 [04:39<00:00,  9.31s/it]


TOP-30 TRANSPORTATION GRAPH PROCESSING COMPLETE


In [9]:
# ============================================================
# CELL 8B: RECOVER AND CLASSIFY FAILED SCENES
# ============================================================

from osmnx._errors import InsufficientResponseError
import requests

processing_log_df = pd.DataFrame(processing_records)
scene_network_metrics_df = pd.DataFrame(scene_metric_records)

print("=" * 72)
print("FAILED-SCENE RECOVERY AND CLASSIFICATION")
print("=" * 72)

failed_scene_ids = processing_log_df.loc[
    processing_log_df["status"] == "FAILED",
    "scene_id",
].tolist()

print(f"\nFailed scenes found: {len(failed_scene_ids)}")
print(failed_scene_ids)

recovery_records = []
recovered_metric_records = []

for scene_id in failed_scene_ids:

    scene = buffered_bounds_gdf.loc[
        buffered_bounds_gdf["scene_id"] == scene_id
    ].iloc[0]

    bbox = (
        float(scene["osm_left"]),
        float(scene["osm_bottom"]),
        float(scene["osm_right"]),
        float(scene["osm_top"]),
    )

    print("\n" + "-" * 72)
    print(f"Reviewing: {scene_id}")

    original_error = processing_log_df.loc[
        processing_log_df["scene_id"] == scene_id,
        "error",
    ].iloc[0]

    # --------------------------------------------------------
    # Cases that already indicate no mapped drive network
    # --------------------------------------------------------

    no_road_indicators = [
        "No data elements in server response",
        "Found no graph nodes within the requested polygon",
    ]

    if any(
        indicator in str(original_error)
        for indicator in no_road_indicators
    ):

        recovery_records.append(
            {
                "scene_id": scene_id,
                "status": "NO_ROAD_DATA",
                "processing_seconds": 0.0,
                "error": original_error,
            }
        )

        recovered_metric_records.append(
            {
                "scene_id": scene_id,
                "country_prefix": scene["country_prefix"],
                "split": scene["split"],
                "quality_rank": scene["quality_rank"],
                "valid_pct": scene["valid_pct"],
                "flood_pct_valid": scene["flood_pct_valid"],
                "permanent_water_pct": scene["permanent_water_pct"],
                "preliminary_suitability_score": (
                    scene["preliminary_suitability_score"]
                ),
                "node_count": 0,
                "directed_edge_count": 0,
                "undirected_edge_count": 0,
                "weak_component_count": 0,
                "strong_component_count": 0,
                "largest_component_nodes": 0,
                "largest_component_share": 0.0,
                "total_road_length_km": 0.0,
                "scene_area_km2": np.nan,
                "road_density_km_per_km2": 0.0,
                "mean_node_degree": 0.0,
                "intersection_count": 0,
                "intersection_density_per_km2": 0.0,
                "dead_end_count": 0,
                "graph_density": 0.0,
                "road_class_count": 0,
                "bridge_edge_count": 0,
                "tunnel_edge_count": 0,
                "named_edge_count": 0,
                "lane_coverage_pct": 0.0,
                "speed_coverage_pct": 0.0,
                "motorway_edge_count": 0,
                "trunk_edge_count": 0,
                "primary_edge_count": 0,
                "secondary_edge_count": 0,
                "tertiary_edge_count": 0,
                "residential_edge_count": 0,
                "unclassified_edge_count": 0,
            }
        )

        print("Classified as NO_ROAD_DATA.")
        continue

    # --------------------------------------------------------
    # Retry temporary request failures
    # --------------------------------------------------------

    print("Retrying temporary request failure...")

    try:

        start_time = time.time()

        graph = ox.graph.graph_from_bbox(
            bbox=bbox,
            network_type="drive",
            simplify=True,
            retain_all=True,
            truncate_by_edge=True,
        )

        nodes_gdf, edges_gdf = ox.convert.graph_to_gdfs(
            graph,
            nodes=True,
            edges=True,
        )

        edges_gdf = edges_gdf.copy()
        nodes_gdf = nodes_gdf.copy()

        edges_gdf["scene_id"] = scene_id

        edges_gdf["highway_clean"] = (
            edges_gdf["highway"].apply(normalize_osm_value)
            if "highway" in edges_gdf.columns
            else None
        )

        edges_gdf["highway_class"] = (
            edges_gdf["highway"].apply(primary_highway_class)
            if "highway" in edges_gdf.columns
            else "unknown"
        )

        edges_gdf["road_importance"] = (
            edges_gdf["highway_class"]
            .apply(classify_road_importance)
        )

        edges_gdf["name_clean"] = (
            edges_gdf["name"].apply(normalize_osm_value)
            if "name" in edges_gdf.columns
            else None
        )

        edges_gdf["lanes_numeric"] = (
            edges_gdf["lanes"].apply(first_numeric_value)
            if "lanes" in edges_gdf.columns
            else np.nan
        )

        edges_gdf["maxspeed_numeric"] = (
            edges_gdf["maxspeed"].apply(first_numeric_value)
            if "maxspeed" in edges_gdf.columns
            else np.nan
        )

        edges_gdf["is_bridge"] = (
            edges_gdf["bridge"].apply(normalize_boolean_tag)
            if "bridge" in edges_gdf.columns
            else False
        )

        edges_gdf["is_tunnel"] = (
            edges_gdf["tunnel"].apply(normalize_boolean_tag)
            if "tunnel" in edges_gdf.columns
            else False
        )

        edges_gdf["is_oneway"] = (
            edges_gdf["oneway"].astype(bool)
            if "oneway" in edges_gdf.columns
            else False
        )

        edges_gdf["length_m"] = pd.to_numeric(
            edges_gdf["length"],
            errors="coerce",
        )

        undirected_graph = ox.convert.to_undirected(graph)
        degree_dict = dict(undirected_graph.degree())

        nodes_gdf["scene_id"] = scene_id
        nodes_gdf["node_degree"] = (
            nodes_gdf.index.map(degree_dict)
            .fillna(0)
            .astype(int)
        )
        nodes_gdf["is_intersection"] = (
            nodes_gdf["node_degree"] >= 3
        )
        nodes_gdf["is_dead_end"] = (
            nodes_gdf["node_degree"] == 1
        )

        scene_polygon = gpd.GeoDataFrame(
            [{"geometry": scene.geometry}],
            crs="EPSG:4326",
        )

        scene_area_km2 = (
            scene_polygon
            .to_crs(scene["local_projected_crs"])
            .geometry.area.iloc[0]
            / 1_000_000
        )

        scene_metrics = build_scene_network_metrics(
            graph=graph,
            nodes_gdf=nodes_gdf,
            edges_gdf=edges_gdf,
            scene_id=scene_id,
            scene_area_km2=scene_area_km2,
        )

        scene_metrics.update(
            {
                "country_prefix": scene["country_prefix"],
                "split": scene["split"],
                "quality_rank": scene["quality_rank"],
                "valid_pct": scene["valid_pct"],
                "flood_pct_valid": scene["flood_pct_valid"],
                "permanent_water_pct": (
                    scene["permanent_water_pct"]
                ),
                "preliminary_suitability_score": (
                    scene["preliminary_suitability_score"]
                ),
            }
        )

        graphml_path = (
            GRAPHML_DIR
            / f"{scene_id}_transportation_graph.graphml"
        )

        gpkg_path = (
            GPKG_DIR
            / f"{scene_id}_transportation_knowledge.gpkg"
        )

        metadata_gdf = gpd.GeoDataFrame(
            [scene_metrics],
            geometry=[scene.geometry],
            crs="EPSG:4326",
        )

        ox.io.save_graphml(
            graph,
            filepath=graphml_path,
        )

        nodes_gdf.to_file(
            gpkg_path,
            layer="nodes",
            driver="GPKG",
        )

        edges_gdf.to_file(
            gpkg_path,
            layer="roads",
            driver="GPKG",
        )

        metadata_gdf.to_file(
            gpkg_path,
            layer="metadata",
            driver="GPKG",
        )

        elapsed_seconds = time.time() - start_time

        recovery_records.append(
            {
                "scene_id": scene_id,
                "status": "RECOVERED",
                "processing_seconds": elapsed_seconds,
                "error": None,
            }
        )

        recovered_metric_records.append(scene_metrics)

        print(
            f"Recovered: {len(nodes_gdf):,} nodes, "
            f"{len(edges_gdf):,} edges."
        )

    except (
        InsufficientResponseError,
        ValueError,
    ) as exc:

        recovery_records.append(
            {
                "scene_id": scene_id,
                "status": "NO_ROAD_DATA",
                "processing_seconds": 0.0,
                "error": str(exc),
            }
        )

        print("Retry returned no usable roadway network.")

    except Exception as exc:

        recovery_records.append(
            {
                "scene_id": scene_id,
                "status": "REQUEST_FAILED",
                "processing_seconds": 0.0,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

        print(
            f"Retry failed again: "
            f"{type(exc).__name__}: {exc}"
        )

recovery_df = pd.DataFrame(recovery_records)
recovered_metrics_df = pd.DataFrame(recovered_metric_records)

print("\nRECOVERY SUMMARY")
print("-" * 72)

print(
    recovery_df["status"]
    .value_counts()
    .to_string()
)

recovery_path = (
    TABLE_DIR
    / "roadflood_vlm_failed_scene_recovery.csv"
)

recovery_df.to_csv(
    recovery_path,
    index=False,
)

print(f"\nSaved to: {recovery_path}")

FAILED-SCENE RECOVERY AND CLASSIFICATION

Failed scenes found: 4
['Paraguay_305760', 'Paraguay_62897', 'Paraguay_605682', 'Somalia_371421']

------------------------------------------------------------------------
Reviewing: Paraguay_305760
Classified as NO_ROAD_DATA.

------------------------------------------------------------------------
Reviewing: Paraguay_62897
Classified as NO_ROAD_DATA.

------------------------------------------------------------------------
Reviewing: Paraguay_605682
Classified as NO_ROAD_DATA.

------------------------------------------------------------------------
Reviewing: Somalia_371421
Classified as NO_ROAD_DATA.

RECOVERY SUMMARY
------------------------------------------------------------------------
status
NO_ROAD_DATA    4

Saved to: /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_failed_scene_recovery.csv


In [10]:
# ============================================================
# CELL 8C: CONSOLIDATE TOP-30 RESULTS
# ============================================================

# Remove the original failed rows and replace them with their
# classified or recovered outcomes.

successful_log_df = processing_log_df.loc[
    processing_log_df["status"] != "FAILED"
].copy()

final_processing_log_df = pd.concat(
    [
        successful_log_df,
        recovery_df,
    ],
    ignore_index=True,
)

final_scene_metrics_df = pd.concat(
    [
        scene_network_metrics_df,
        recovered_metrics_df,
    ],
    ignore_index=True,
)

# Ensure exactly one record per scene.
final_processing_log_df = (
    final_processing_log_df
    .drop_duplicates(
        subset="scene_id",
        keep="last",
    )
)

final_scene_metrics_df = (
    final_scene_metrics_df
    .drop_duplicates(
        subset="scene_id",
        keep="last",
    )
)

print("=" * 72)
print("CONSOLIDATED TOP-30 RESULTS")
print("=" * 72)

print("\nFinal processing status:")
print(
    final_processing_log_df["status"]
    .value_counts()
    .to_string()
)

print(
    f"\nScenes represented in metrics: "
    f"{len(final_scene_metrics_df):,}/30"
)

final_metrics_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_metrics.csv"
)

final_log_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_processing_log.csv"
)

final_scene_metrics_df.to_csv(
    final_metrics_path,
    index=False,
)

final_processing_log_df.to_csv(
    final_log_path,
    index=False,
)

print(f"\nMetrics saved to: {final_metrics_path}")
print(f"Log saved to    : {final_log_path}")

CONSOLIDATED TOP-30 RESULTS

Final processing status:
status
LOADED_EXISTING    26
NO_ROAD_DATA        4

Scenes represented in metrics: 30/30

Metrics saved to: /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_top30_transportation_metrics.csv
Log saved to    : /home/adjeiowusu1/myproject/ResilientVLM/outputs/tables/roadflood_vlm_top30_transportation_processing_log.csv


In [11]:
# ============================================================
# CELL 9: SAVE, VALIDATE, AND SUMMARIZE FINAL TOP-30 RESULTS
# ============================================================

print("=" * 72)
print("FINAL TRANSPORTATION KNOWLEDGE GRAPH SUMMARY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Use the consolidated outputs from Cell 8C
# ------------------------------------------------------------

scene_network_metrics_df = final_scene_metrics_df.copy()
processing_log_df = final_processing_log_df.copy()

# ------------------------------------------------------------
# 2. Validate required objects and scene uniqueness
# ------------------------------------------------------------

if scene_network_metrics_df.empty:
    raise ValueError(
        "The consolidated transportation metrics table is empty. "
        "Run Cells 8, 8B, and 8C before running Cell 9."
    )

if processing_log_df.empty:
    raise ValueError(
        "The consolidated processing log is empty. "
        "Run Cells 8, 8B, and 8C before running Cell 9."
    )

# Retain only one record per scene in case a recovery cell
# was executed more than once.
scene_network_metrics_df = (
    scene_network_metrics_df
    .drop_duplicates(
        subset="scene_id",
        keep="last",
    )
    .reset_index(drop=True)
)

processing_log_df = (
    processing_log_df
    .drop_duplicates(
        subset="scene_id",
        keep="last",
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Standardize processing statuses
# ------------------------------------------------------------

status_map = {
    "DOWNLOADED": "SUCCESS",
    "LOADED_EXISTING": "SUCCESS",
    "RECOVERED": "SUCCESS",
    "NO_ROAD_DATA": "NO_ROAD_DATA",
    "REQUEST_FAILED": "REQUEST_FAILED",
    "FAILED": "REQUEST_FAILED",
}

processing_log_df["final_status"] = (
    processing_log_df["status"]
    .map(status_map)
    .fillna(processing_log_df["status"])
)

# Add final status to the metrics table.
scene_network_metrics_df = (
    scene_network_metrics_df
    .drop(
        columns=["final_status"],
        errors="ignore",
    )
    .merge(
        processing_log_df[
            [
                "scene_id",
                "final_status",
            ]
        ],
        on="scene_id",
        how="left",
    )
)

# Any scene represented by valid positive graph metrics should be
# treated as successful, even if the merged status is missing.
successful_metric_mask = (
    (
        pd.to_numeric(
            scene_network_metrics_df["node_count"],
            errors="coerce",
        ).fillna(0) > 0
    )
    &
    (
        pd.to_numeric(
            scene_network_metrics_df["directed_edge_count"],
            errors="coerce",
        ).fillna(0) > 0
    )
)

scene_network_metrics_df.loc[
    successful_metric_mask
    &
    scene_network_metrics_df["final_status"].isna(),
    "final_status",
] = "SUCCESS"

scene_network_metrics_df["final_status"] = (
    scene_network_metrics_df["final_status"]
    .fillna("UNKNOWN")
)

# ------------------------------------------------------------
# 4. Ensure numeric metric columns are numeric
# ------------------------------------------------------------

numeric_columns = [
    "quality_rank",
    "valid_pct",
    "flood_pct_valid",
    "permanent_water_pct",
    "preliminary_suitability_score",
    "node_count",
    "directed_edge_count",
    "undirected_edge_count",
    "weak_component_count",
    "strong_component_count",
    "largest_component_nodes",
    "largest_component_share",
    "total_road_length_km",
    "scene_area_km2",
    "road_density_km_per_km2",
    "mean_node_degree",
    "intersection_count",
    "intersection_density_per_km2",
    "dead_end_count",
    "graph_density",
    "road_class_count",
    "bridge_edge_count",
    "tunnel_edge_count",
    "named_edge_count",
    "lane_coverage_pct",
    "speed_coverage_pct",
    "motorway_edge_count",
    "trunk_edge_count",
    "primary_edge_count",
    "secondary_edge_count",
    "tertiary_edge_count",
    "residential_edge_count",
    "unclassified_edge_count",
]

for column in numeric_columns:
    if column in scene_network_metrics_df.columns:
        scene_network_metrics_df[column] = pd.to_numeric(
            scene_network_metrics_df[column],
            errors="coerce",
        )

# ------------------------------------------------------------
# 5. Calculate transportation usability fields
# ------------------------------------------------------------

scene_network_metrics_df["has_road_network"] = (
    (
        scene_network_metrics_df["node_count"]
        .fillna(0)
        > 0
    )
    &
    (
        scene_network_metrics_df["directed_edge_count"]
        .fillna(0)
        > 0
    )
)

scene_network_metrics_df["has_intersections"] = (
    scene_network_metrics_df["intersection_count"]
    .fillna(0)
    > 0
)

scene_network_metrics_df["has_bridge"] = (
    scene_network_metrics_df["bridge_edge_count"]
    .fillna(0)
    > 0
)

scene_network_metrics_df["has_major_road"] = (
    scene_network_metrics_df[
        [
            "motorway_edge_count",
            "trunk_edge_count",
            "primary_edge_count",
            "secondary_edge_count",
        ]
    ]
    .fillna(0)
    .sum(axis=1)
    > 0
)

# A preliminary transportation usability rule.
# This is not yet the final Transportation Suitability Score.
scene_network_metrics_df["transportation_usable"] = (
    scene_network_metrics_df["has_road_network"]
    &
    (
        scene_network_metrics_df["total_road_length_km"]
        .fillna(0)
        > 0
    )
    &
    (
        scene_network_metrics_df["road_class_count"]
        .fillna(0)
        >= 1
    )
)

# ------------------------------------------------------------
# 6. Sort the final tables
# ------------------------------------------------------------

scene_network_metrics_df = (
    scene_network_metrics_df
    .sort_values(
        [
            "transportation_usable",
            "road_density_km_per_km2",
            "preliminary_suitability_score",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

processing_log_df = (
    processing_log_df
    .sort_values(
        [
            "final_status",
            "scene_id",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Save final consolidated outputs
# ------------------------------------------------------------

scene_metrics_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_metrics.csv"
)

processing_log_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_processing_log.csv"
)

usable_scene_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_transportation_usable_scenes.csv"
)

no_road_scene_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_no_road_scenes.csv"
)

request_failure_path = (
    TABLE_DIR
    / "roadflood_vlm_top30_request_failures.csv"
)

scene_network_metrics_df.to_csv(
    scene_metrics_path,
    index=False,
)

processing_log_df.to_csv(
    processing_log_path,
    index=False,
)

scene_network_metrics_df.loc[
    scene_network_metrics_df["transportation_usable"]
].to_csv(
    usable_scene_path,
    index=False,
)

scene_network_metrics_df.loc[
    scene_network_metrics_df["final_status"]
    == "NO_ROAD_DATA"
].to_csv(
    no_road_scene_path,
    index=False,
)

processing_log_df.loc[
    processing_log_df["final_status"]
    == "REQUEST_FAILED"
].to_csv(
    request_failure_path,
    index=False,
)

# ------------------------------------------------------------
# 8. Processing-status summary
# ------------------------------------------------------------

print("\nPROCESSING STATUS")
print("-" * 72)

status_summary = (
    processing_log_df["final_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="scene_count")
)

status_summary["percentage"] = (
    100
    * status_summary["scene_count"]
    / len(processing_log_df)
)

print(
    status_summary
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. Dataset coverage summary
# ------------------------------------------------------------

total_selected = len(top_scenes_df)
total_log_records = len(processing_log_df)
total_metric_records = len(scene_network_metrics_df)

successful_scenes = int(
    (
        processing_log_df["final_status"]
        == "SUCCESS"
    ).sum()
)

no_road_scenes = int(
    (
        processing_log_df["final_status"]
        == "NO_ROAD_DATA"
    ).sum()
)

request_failed_scenes = int(
    (
        processing_log_df["final_status"]
        == "REQUEST_FAILED"
    ).sum()
)

usable_scenes = int(
    scene_network_metrics_df[
        "transportation_usable"
    ].sum()
)

print("\nDATASET COVERAGE")
print("-" * 72)

print(f"Top-ranked scenes selected       : {total_selected:,}")
print(f"Scenes in final processing log   : {total_log_records:,}")
print(f"Scenes represented in metrics    : {total_metric_records:,}")
print(f"Successful road extractions      : {successful_scenes:,}")
print(f"No mapped drive-network scenes   : {no_road_scenes:,}")
print(f"Remaining request failures       : {request_failed_scenes:,}")
print(f"Transportation-usable scenes     : {usable_scenes:,}")

if total_selected > 0:
    print(
        f"Transportation usability rate    : "
        f"{100 * usable_scenes / total_selected:.1f}%"
    )

# ------------------------------------------------------------
# 10. Network metric summary for usable scenes only
# ------------------------------------------------------------

usable_metrics_df = scene_network_metrics_df.loc[
    scene_network_metrics_df["transportation_usable"]
].copy()

summary_columns = [
    "node_count",
    "directed_edge_count",
    "total_road_length_km",
    "road_density_km_per_km2",
    "intersection_count",
    "intersection_density_per_km2",
    "weak_component_count",
    "largest_component_share",
    "road_class_count",
    "bridge_edge_count",
    "lane_coverage_pct",
    "speed_coverage_pct",
]

available_summary_columns = [
    column
    for column in summary_columns
    if column in usable_metrics_df.columns
]

print("\nNETWORK METRIC SUMMARY — USABLE SCENES")
print("-" * 72)

if usable_metrics_df.empty:
    print("No transportation-usable scenes were found.")

else:
    print(
        usable_metrics_df[
            available_summary_columns
        ]
        .describe()
        .round(3)
        .to_string()
    )

# ------------------------------------------------------------
# 11. Top scenes by roadway density
# ------------------------------------------------------------

print("\nTOP 15 SCENES BY ROAD DENSITY")
print("-" * 72)

road_density_display_columns = [
    "scene_id",
    "country_prefix",
    "split",
    "quality_rank",
    "node_count",
    "directed_edge_count",
    "total_road_length_km",
    "road_density_km_per_km2",
    "intersection_count",
    "intersection_density_per_km2",
    "road_class_count",
    "bridge_edge_count",
]

available_density_columns = [
    column
    for column in road_density_display_columns
    if column in usable_metrics_df.columns
]

if usable_metrics_df.empty:
    print("No scenes are available for roadway-density ranking.")

else:
    print(
        usable_metrics_df[
            available_density_columns
        ]
        .sort_values(
            "road_density_km_per_km2",
            ascending=False,
        )
        .head(15)
        .round(3)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 12. Top scenes by intersection density
# ------------------------------------------------------------

print("\nTOP 15 SCENES BY INTERSECTION DENSITY")
print("-" * 72)

intersection_display_columns = [
    "scene_id",
    "country_prefix",
    "split",
    "quality_rank",
    "road_density_km_per_km2",
    "intersection_count",
    "intersection_density_per_km2",
    "weak_component_count",
    "largest_component_share",
    "road_class_count",
]

available_intersection_columns = [
    column
    for column in intersection_display_columns
    if column in usable_metrics_df.columns
]

if usable_metrics_df.empty:
    print("No scenes are available for intersection-density ranking.")

else:
    print(
        usable_metrics_df[
            available_intersection_columns
        ]
        .sort_values(
            "intersection_density_per_km2",
            ascending=False,
        )
        .head(15)
        .round(3)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 13. Scenes containing major roads or bridge infrastructure
# ------------------------------------------------------------

print("\nCRITICAL TRANSPORTATION INFRASTRUCTURE SCENES")
print("-" * 72)

critical_scene_columns = [
    "scene_id",
    "country_prefix",
    "split",
    "quality_rank",
    "motorway_edge_count",
    "trunk_edge_count",
    "primary_edge_count",
    "secondary_edge_count",
    "bridge_edge_count",
    "total_road_length_km",
    "road_density_km_per_km2",
]

available_critical_columns = [
    column
    for column in critical_scene_columns
    if column in usable_metrics_df.columns
]

critical_scenes_df = usable_metrics_df.loc[
    usable_metrics_df["has_major_road"]
    |
    usable_metrics_df["has_bridge"]
].copy()

if critical_scenes_df.empty:
    print(
        "No usable scenes contain mapped major-road "
        "or bridge infrastructure."
    )

else:
    print(
        critical_scenes_df[
            available_critical_columns
        ]
        .sort_values(
            [
                "bridge_edge_count",
                "road_density_km_per_km2",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .round(3)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 14. No-road and unresolved scenes
# ------------------------------------------------------------

print("\nSCENES WITH NO MAPPED DRIVE NETWORK")
print("-" * 72)

no_road_ids = processing_log_df.loc[
    processing_log_df["final_status"]
    == "NO_ROAD_DATA",
    "scene_id",
].tolist()

if no_road_ids:
    for scene_id in no_road_ids:
        print(scene_id)
else:
    print("None")

print("\nUNRESOLVED REQUEST FAILURES")
print("-" * 72)

failed_rows = processing_log_df.loc[
    processing_log_df["final_status"]
    == "REQUEST_FAILED",
    [
        "scene_id",
        "error",
    ],
]

if failed_rows.empty:
    print("None")
else:
    print(
        failed_rows.to_string(index=False)
    )

# ------------------------------------------------------------
# 15. Final output paths
# ------------------------------------------------------------

print("\nOUTPUT FILES")
print("-" * 72)

print(f"Final metrics table       : {scene_metrics_path}")
print(f"Final processing log      : {processing_log_path}")
print(f"Usable scene table        : {usable_scene_path}")
print(f"No-road scene table       : {no_road_scene_path}")
print(f"Request failure table     : {request_failure_path}")
print(f"GraphML directory         : {GRAPHML_DIR}")
print(f"GeoPackage directory      : {GPKG_DIR}")

print("\n" + "=" * 72)
print("FINAL TOP-30 TRANSPORTATION SUMMARY COMPLETE")
print("=" * 72)

FINAL TRANSPORTATION KNOWLEDGE GRAPH SUMMARY

PROCESSING STATUS
------------------------------------------------------------------------
      status  scene_count  percentage
     SUCCESS           26       86.67
NO_ROAD_DATA            4       13.33

DATASET COVERAGE
------------------------------------------------------------------------
Top-ranked scenes selected       : 30
Scenes in final processing log   : 30
Scenes represented in metrics    : 30
Successful road extractions      : 26
No mapped drive-network scenes   : 4
Remaining request failures       : 0
Transportation-usable scenes     : 26
Transportation usability rate    : 86.7%

NETWORK METRIC SUMMARY — USABLE SCENES
------------------------------------------------------------------------
       node_count  directed_edge_count  total_road_length_km  road_density_km_per_km2  intersection_count  intersection_density_per_km2  weak_component_count  largest_component_share  road_class_count  bridge_edge_count  lane_coverage_pct  